In [ ]:
# Install dependencies
!pip install tensorflow numpy pandas scikit-learn matplotlib beautifulsoup4 lxml nltk -q


In [ ]:
# Clone data repo
!git clone https://github.com/Aditya-dom/legaldocsvalution.git /content/legaldocsvalution -q


In [ ]:
# Build data.csv (texts) and data1.csv (phrase summaries) from the dataset
import os

text_dir = "/content/legaldocsvalution/dataset/indian_data/ind_text"
phrase_dir = "/content/legaldocsvalution/dataset/indian_data/ind_phrases_2"

texts, summaries = [], []
for i in range(1, 94):
    # Full text
    txt_path = os.path.join(text_dir, f"{i}.txt")
    with open(txt_path, "r", errors="ignore") as f:
        texts.append(f.read().replace("\n", " ").strip())
    # Phrase summary: extract first-column phrases, join as summary
    phrase_file = os.path.join(phrase_dir, f"{i}_2.txt")
    if os.path.exists(phrase_file):
        with open(phrase_file, "r", errors="ignore") as f:
            phrases = [line.rsplit(None, 4)[0].strip() for line in f if line.strip()]
        summaries.append(" . ".join(phrases))
    else:
        summaries.append(texts[-1][:500])

with open("/content/data.csv", "w") as f:
    f.write("Text\n")
    for t in texts:
        f.write(t.replace(",", " ") + "\n")

with open("/content/data1.csv", "w") as f:
    for s in summaries:
        f.write(s.replace(",", " ") + "\n")

print(f"Created data.csv ({len(texts)} rows) and data1.csv ({len(summaries)} rows)")


In [ ]:
import pandas as pd
import numpy as np


In [ ]:
a = []
with open("/content/data.csv", "r", errors="ignore") as f:
    for line in f:
        a.append(line)


In [ ]:
df = pd.DataFrame(a, columns=["Text"])
df


In [ ]:
b = []
with open("/content/data1.csv", "r", errors="ignore") as f:
    for line in f:
        b.append(line)


In [ ]:
df1 = pd.DataFrame(b)
df1


In [ ]:
df["Summary"] = df1
df


In [ ]:
df = df.loc[60:65]

a = df["Text"]
b = df["Summary"]
a = np.array(a)
b = np.array(b)
df2 = pd.DataFrame(a, columns=["Text"])
df2["Summary"] = b
df2


In [ ]:
import tensorflow as tf
import os
from tensorflow.keras.layers import Layer
from tensorflow.keras import backend as K


class AttentionLayer(Layer):
    """
    Bahdanau attention (https://arxiv.org/pdf/1409.0473.pdf).
    """

    def __init__(self, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        assert isinstance(input_shape, list)
        self.W_a = self.add_weight(name="W_a",
                                   shape=tf.TensorShape((input_shape[0][2], input_shape[0][2])),
                                   initializer="uniform", trainable=True)
        self.U_a = self.add_weight(name="U_a",
                                   shape=tf.TensorShape((input_shape[1][2], input_shape[0][2])),
                                   initializer="uniform", trainable=True)
        self.V_a = self.add_weight(name="V_a",
                                   shape=tf.TensorShape((input_shape[0][2], 1)),
                                   initializer="uniform", trainable=True)
        super(AttentionLayer, self).build(input_shape)

    def call(self, inputs, verbose=False):
        assert type(inputs) == list
        encoder_out_seq, decoder_out_seq = inputs

        def energy_step(inputs, states):
            en_seq_len, en_hidden = encoder_out_seq.shape[1], encoder_out_seq.shape[2]
            reshaped_enc_outputs = K.reshape(encoder_out_seq, (-1, en_hidden))
            W_a_dot_s = K.reshape(K.dot(reshaped_enc_outputs, self.W_a), (-1, en_seq_len, en_hidden))
            U_a_dot_h = K.expand_dims(K.dot(inputs, self.U_a), 1)
            reshaped_Ws_plus_Uh = K.tanh(K.reshape(W_a_dot_s + U_a_dot_h, (-1, en_hidden)))
            e_i = K.reshape(K.dot(reshaped_Ws_plus_Uh, self.V_a), (-1, en_seq_len))
            e_i = K.softmax(e_i)
            return e_i, [e_i]

        def context_step(inputs, states):
            c_i = K.sum(encoder_out_seq * K.expand_dims(inputs, -1), axis=1)
            return c_i, [c_i]

        def create_inital_state(inputs, hidden_size):
            fake_state = K.zeros_like(inputs)
            fake_state = K.sum(fake_state, axis=[1, 2])
            fake_state = K.expand_dims(fake_state)
            fake_state = K.tile(fake_state, [1, hidden_size])
            return fake_state

        fake_state_c = create_inital_state(encoder_out_seq, encoder_out_seq.shape[-1])
        fake_state_e = create_inital_state(encoder_out_seq, encoder_out_seq.shape[1])
        last_out, e_outputs, _ = K.rnn(energy_step, decoder_out_seq, [fake_state_e])
        last_out, c_outputs, _ = K.rnn(context_step, e_outputs, [fake_state_c])
        return c_outputs, e_outputs

    def compute_output_shape(self, input_shape):
        return [
            tf.TensorShape((input_shape[1][0], input_shape[1][1], input_shape[1][2])),
            tf.TensorShape((input_shape[1][0], input_shape[1][1], input_shape[0][1]))
        ]


In [ ]:
import numpy as np
import pandas as pd
import re
from bs4 import BeautifulSoup
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from nltk.corpus import stopwords
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense, Concatenate, TimeDistributed
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
import warnings
import nltk
nltk.download("stopwords", quiet=True)
pd.set_option("display.max_colwidth", 200)
warnings.filterwarnings("ignore")


In [ ]:
data = df2
data


In [ ]:
contraction_mapping = {"ain't": "is not", "aren't": "are not", "can't": "cannot",
    "'cause": "because", "could've": "could have", "couldn't": "could not",
    "didn't": "did not", "doesn't": "does not", "don't": "do not",
    "hadn't": "had not", "hasn't": "has not", "haven't": "have not",
    "he'd": "he would", "he'll": "he will", "he's": "he is",
    "I'd": "I would", "I'll": "I will", "I'm": "I am", "I've": "I have",
    "i'd": "i would", "i'll": "i will", "i'm": "i am", "i've": "i have",
    "isn't": "is not", "it's": "it is", "let's": "let us",
    "she'd": "she would", "she'll": "she will", "she's": "she is",
    "should've": "should have", "shouldn't": "should not",
    "that's": "that is", "there's": "there is", "they'd": "they would",
    "they'll": "they will", "they're": "they are", "they've": "they have",
    "wasn't": "was not", "we'd": "we would", "we'll": "we will",
    "we're": "we are", "we've": "we have", "weren't": "were not",
    "what's": "what is", "who's": "who is", "won't": "will not",
    "would've": "would have", "wouldn't": "would not",
    "you'd": "you would", "you'll": "you will",
    "you're": "you are", "you've": "you have"}


In [ ]:
stop_words = set(stopwords.words("english"))

def text_cleaner(text):
    newString = text.lower()
    newString = BeautifulSoup(newString, "lxml").text
    newString = re.sub(r"\([^)]*\)", "", newString)
    newString = re.sub('"', "", newString)
    newString = " ".join([contraction_mapping[t] if t in contraction_mapping else t
                          for t in newString.split(" ")])
    newString = re.sub(r"'s\b", "", newString)
    newString = re.sub("[^a-zA-Z]", " ", newString)
    tokens = [w for w in newString.split() if w not in stop_words]
    return " ".join([i for i in tokens if len(i) >= 3]).strip()

cleaned_text = [text_cleaner(t) for t in data["Text"]]


In [ ]:
def summary_cleaner(text):
    newString = re.sub('"', "", str(text))
    newString = " ".join([contraction_mapping[t] if t in contraction_mapping else t
                          for t in newString.split(" ")])
    newString = re.sub(r"'s\b", "", newString)
    newString = re.sub("[^a-zA-Z]", " ", newString)
    newString = newString.lower()
    return " ".join([i for i in newString.split() if len(i) > 1])

cleaned_summary = [summary_cleaner(t) for t in data["Summary"]]

data["cleaned_text"] = cleaned_text
data["cleaned_summary"] = cleaned_summary
data["cleaned_summary"].replace("", np.nan, inplace=True)
data.dropna(axis=0, inplace=True)


In [ ]:
data["cleaned_summary"] = data["cleaned_summary"].apply(
    lambda x: "_START_ " + x + " _END_")


In [ ]:
import matplotlib.pyplot as plt
text_word_count = [len(i.split()) for i in data["cleaned_text"]]
summary_word_count = [len(i.split()) for i in data["cleaned_summary"]]
length_df = pd.DataFrame({"text": text_word_count, "summary": summary_word_count})
length_df.hist(bins=30)
plt.show()


In [ ]:
max_len_text = 3000
max_len_summary = 400


In [ ]:
from sklearn.model_selection import train_test_split
x_tr, x_val, y_tr, y_val = train_test_split(
    data["cleaned_text"], data["cleaned_summary"],
    test_size=0.1, random_state=0, shuffle=True)


In [ ]:
x_tokenizer = Tokenizer()
x_tokenizer.fit_on_texts(list(x_tr))
x_tr  = pad_sequences(x_tokenizer.texts_to_sequences(x_tr),  maxlen=max_len_text,    padding="post")
x_val = pad_sequences(x_tokenizer.texts_to_sequences(x_val), maxlen=max_len_text,    padding="post")
x_voc_size = len(x_tokenizer.word_index) + 1


In [ ]:
y_tokenizer = Tokenizer()
y_tokenizer.fit_on_texts(list(y_tr))
y_tr  = pad_sequences(y_tokenizer.texts_to_sequences(y_tr),  maxlen=max_len_summary, padding="post")
y_val = pad_sequences(y_tokenizer.texts_to_sequences(y_val), maxlen=max_len_summary, padding="post")
y_voc_size = len(y_tokenizer.word_index) + 1


In [ ]:
from tensorflow.keras import backend as K
K.clear_session()
latent_dim = 500

encoder_inputs = Input(shape=(max_len_text,))
enc_emb = Embedding(x_voc_size, latent_dim, trainable=True)(encoder_inputs)
encoder_lstm1 = LSTM(latent_dim, return_sequences=True, return_state=True)
encoder_output1, state_h1, state_c1 = encoder_lstm1(enc_emb)
encoder_lstm2 = LSTM(latent_dim, return_sequences=True, return_state=True)
encoder_output2, state_h2, state_c2 = encoder_lstm2(encoder_output1)
encoder_lstm3 = LSTM(latent_dim, return_state=True, return_sequences=True)
encoder_outputs, state_h, state_c = encoder_lstm3(encoder_output2)

decoder_inputs = Input(shape=(None,))
dec_emb_layer = Embedding(y_voc_size, latent_dim, trainable=True)
dec_emb = dec_emb_layer(decoder_inputs)
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, decoder_fwd_state, decoder_back_state = decoder_lstm(
    dec_emb, initial_state=[state_h, state_c])

attn_layer = AttentionLayer(name="attention_layer")
attn_out, attn_states = attn_layer([encoder_outputs, decoder_outputs])
decoder_concat_input = Concatenate(axis=-1, name="concat_layer")([decoder_outputs, attn_out])
decoder_dense = TimeDistributed(Dense(y_voc_size, activation="softmax"))
decoder_outputs = decoder_dense(decoder_concat_input)

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.summary()


In [ ]:
model.compile(optimizer="rmsprop", loss="sparse_categorical_crossentropy")


In [ ]:
es = EarlyStopping(monitor="val_loss", mode="min", verbose=1)
history = model.fit(
    [x_tr, y_tr[:, :-1]],
    y_tr.reshape(y_tr.shape[0], y_tr.shape[1], 1)[:, 1:],
    epochs=50,
    callbacks=[es],
    batch_size=64,
    validation_data=(
        [x_val, y_val[:, :-1]],
        y_val.reshape(y_val.shape[0], y_val.shape[1], 1)[:, 1:]
    )
)


In [ ]:
from matplotlib import pyplot
pyplot.plot(history.history["loss"], label="train")
pyplot.plot(history.history["val_loss"], label="val")
pyplot.legend()
pyplot.show()


In [ ]:
reverse_target_word_index = y_tokenizer.index_word
reverse_source_word_index = x_tokenizer.index_word
target_word_index = y_tokenizer.word_index


In [ ]:
encoder_model = Model(inputs=encoder_inputs, outputs=[encoder_outputs, state_h, state_c])

decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_hidden_state_input = Input(shape=(max_len_text, latent_dim))
dec_emb2 = dec_emb_layer(decoder_inputs)
decoder_outputs2, state_h2, state_c2 = decoder_lstm(
    dec_emb2, initial_state=[decoder_state_input_h, decoder_state_input_c])
attn_out_inf, attn_states_inf = attn_layer([decoder_hidden_state_input, decoder_outputs2])
decoder_inf_concat = Concatenate(axis=-1, name="concat")([decoder_outputs2, attn_out_inf])
decoder_outputs2 = decoder_dense(decoder_inf_concat)
decoder_model = Model(
    [decoder_inputs] + [decoder_hidden_state_input, decoder_state_input_h, decoder_state_input_c],
    [decoder_outputs2] + [state_h2, state_c2])


In [ ]:
def decode_sequence(input_seq):
    e_out, e_h, e_c = encoder_model.predict(input_seq, verbose=0)
    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = target_word_index.get("start", 1)
    stop_condition = False
    decoded_sentence = ""
    while not stop_condition:
        output_tokens, h, c = decoder_model.predict(
            [target_seq] + [e_out, e_h, e_c], verbose=0)
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_token = reverse_target_word_index.get(sampled_token_index, "")
        if sampled_token != "end":
            decoded_sentence += " " + sampled_token
        if sampled_token == "end" or len(decoded_sentence.split()) >= (max_len_summary - 1):
            stop_condition = True
        target_seq = np.zeros((1, 1))
        target_seq[0, 0] = sampled_token_index
        e_h, e_c = h, c
    return decoded_sentence


In [ ]:
def seq2summary(input_seq):
    return " ".join([reverse_target_word_index[i] for i in input_seq
                     if i != 0
                     and i != target_word_index.get("start")
                     and i != target_word_index.get("end")])

def seq2text(input_seq):
    return " ".join([reverse_source_word_index[i] for i in input_seq if i != 0])


In [ ]:
for i in range(len(x_val)):
    print("Review:", seq2text(x_val[i]))
    print("Original summary:", seq2summary(y_val[i]))
    print("Predicted summary:", decode_sequence(x_val[i].reshape(1, max_len_text)))
    print()
